In [1]:
import pandas as pd
import numpy as np

In [2]:
dataset_1 = pd.read_csv('../data/Train data/2024-09-08_2024-09-14_Win64_train_with_nulls.csv')
dataset_2 = pd.read_csv('../data/Train data/2024-09-15_2024-09-21_Win64_train_with_nulls.csv')

In [3]:
dataset_1.shape

(20000, 32)

In [4]:
dataset_2.shape

(20000, 32)

In [5]:
dataset_1['family'].isnull().sum()

np.int64(12072)

In [6]:
dataset_1['family_confidence'].isnull().sum()

np.int64(12072)

family                   12072
family_confidence        12072

In [7]:
dataset_1.columns

Index(['md5', 'sha1', 'sha256', 'tlsh', 'first_submission_date',
       'last_analysis_date', 'detection_ratio', 'label', 'file_type', 'family',
       'family_confidence', 'behavior', 'file_property', 'packer', 'exploit',
       'group', 'histogram', 'byteentropy', 'strings', 'general', 'header',
       'section', 'imports', 'exports', 'datadirectories', 'richheader',
       'authenticode', 'pefilewarnings', 'week_id', 'caps', 'ttps', 'mbc'],
      dtype='str')

In [39]:
dataset_1['sha256'].describe(include=all)

/var/folders/m5/6qc5jvtd7jq6xr1n9xf984rr0000gn/T/ipykernel_14712/1643659544.py:1: Pandas4Warning: The 'include' and 'exclude' arguments are deprecated for Series.describe and will be removed in a future version. These arguments have no effect on Series and will be removed.
  dataset_1['sha256'].describe(include=all)


count                                                 20000
unique                                                10000
top       00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...
freq                                                      2
Name: sha256, dtype: object

In [42]:
dataset_1['sha256'].value_counts().value_counts()

count
2    10000
Name: count, dtype: int64

In [44]:
duplicate_rows = dataset_1[
    dataset_1['sha256'].duplicated(keep=False)
]

print(
    duplicate_rows.groupby('sha256')
    .nunique()
    .describe()
)

           md5     sha1     tlsh  first_submission_date  last_analysis_date  \
count  10000.0  10000.0  10000.0                10000.0             10000.0   
mean       1.0      1.0      1.0                    1.0                 1.0   
std        0.0      0.0      0.0                    0.0                 0.0   
min        1.0      1.0      1.0                    1.0                 1.0   
25%        1.0      1.0      1.0                    1.0                 1.0   
50%        1.0      1.0      1.0                    1.0                 1.0   
75%        1.0      1.0      1.0                    1.0                 1.0   
max        1.0      1.0      1.0                    1.0                 1.0   

       detection_ratio    label  file_type        family  family_confidence  \
count          10000.0  10000.0    10000.0  10000.000000       10000.000000   
mean               1.0      1.0        1.0      0.396400           0.396400   
std                0.0      0.0        0.0      0.4

In [43]:
print(dataset_1.duplicated().sum())

2958


In [40]:
dataset_1['sha256'].value_counts().idxmax()

'00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2'

In [45]:
duplicate_rows = dataset_1[
    dataset_1['sha256'].duplicated(keep=False)
]

variation = (
    duplicate_rows.groupby('sha256')
    .nunique()
)

print(
    variation.gt(1).sum()
    .sort_values(ascending=False)
)

caps                     7019
mbc                      6591
ttps                     6276
byteentropy                 0
week_id                     0
pefilewarnings              0
authenticode                0
richheader                  0
datadirectories             0
exports                     0
imports                     0
section                     0
header                      0
general                     0
strings                     0
md5                         0
sha1                        0
group                       0
exploit                     0
packer                      0
file_property               0
behavior                    0
family_confidence           0
family                      0
file_type                   0
label                       0
detection_ratio             0
last_analysis_date          0
first_submission_date       0
tlsh                        0
histogram                   0
dtype: int64


In [46]:
label_conflicts = (
    duplicate_rows.groupby('sha256')['label']
    .nunique()
)

print("Conflicting labels:",
      (label_conflicts > 1).sum())

Conflicting labels: 0


In [47]:
for col in ['md5', 'sha1']:
    result = duplicate_rows.groupby('sha256')[col].nunique()

    print(col)
    print("Conflicting groups:", (result > 1).sum())

md5
Conflicting groups: 0
sha1
Conflicting groups: 0


In [48]:
sample_hash = variation[
    variation.gt(1).any(axis=1)
].index[0]

print(
    duplicate_rows[
        duplicate_rows['sha256'] == sample_hash
    ].T
)

                                                                   0      \
md5                                     9c6701567cbcf50e11810c0a1ca2a580   
sha1                            43e53f8e44ecd49b55c8f5ed9ae33b8dd8e9534d   
sha256                 00047dd8d224e115c979a9eb1acd116e7a4a8a72f02594...   
tlsh                   T1B9C6338D77150C9DE95FA239E4E1863766B375B50BA0...   
first_submission_date                                         1726320821   
last_analysis_date                                            1730346557   
detection_ratio                                                    12/73   
label                                                                  1   
file_type                                                          Win64   
family                                                               NaN   
family_confidence                                                    NaN   
behavior                                                              []   
file_propert

In [55]:
# Check how many records have empty/non-empty
# CAPS, TTPS and MBC values

for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{col.upper()}")

    print("Empty:",
          dataset_1[col].apply(
              lambda x: len(x) == 0 if isinstance(x, list) else pd.isna(x)
          ).sum())

    print("Non-empty:",
          dataset_1[col].apply(
              lambda x: len(x) > 0 if isinstance(x, list) else False
          ).sum())


CAPS
Empty: 0
Non-empty: 0

TTPS
Empty: 0
Non-empty: 0

MBC
Empty: 0
Non-empty: 0


In [56]:
for col in ['caps', 'ttps', 'mbc']:

    print("\nCOLUMN:", col)

    print("Data types:")
    print(dataset_1[col].apply(type).value_counts())

    print("\nSample values:")
    print(dataset_1[col].head(5).tolist())


COLUMN: caps
Data types:
caps
<class 'str'>    20000
Name: count, dtype: int64

Sample values:
['[]', "[{'Capability': 'Find graphical window', 'Namespace': 'host-interaction/gui/window/find', 'Addrs': []}, {'Capability': 'Create or open mutex on windows', 'Namespace': 'host-interaction/mutex', 'Addrs': []}, {'Capability': 'Link function at runtime on windows', 'Namespace': 'linking/runtime-linking', 'Addrs': []}, {'Capability': 'Encode data using xor', 'Namespace': 'data-manipulation/encoding/xor', 'Addrs': ['0x140017f30', '0x140017f60', '0x140017f90', '0x140017fc0', '0x140017ff0', '0x140018030', '0x14006b5a0']}, {'Capability': 'Get file attributes', 'Namespace': 'host-interaction/file-system/meta', 'Addrs': ['0x14007c500']}, {'Capability': 'Create thread', 'Namespace': 'host-interaction/thread/create', 'Addrs': ['0x14007d3c0']}, {'Capability': 'Accept command line arguments', 'Namespace': 'host-interaction/cli', 'Addrs': ['0x140002cd0']}, {'Capability': 'Check mutex on windows', 'Na

In [57]:
import ast
import pandas as pd

for col in ['caps', 'ttps', 'mbc']:

    dataset_1[col] = dataset_1[col].apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str) else x
    )

    empty = dataset_1[col].apply(
        lambda x: len(x) == 0
    ).sum()

    non_empty = dataset_1[col].apply(
        lambda x: len(x) > 0
    ).sum()

    print(f"\n{col.upper()}")
    print("Empty:", empty)
    print("Non-empty:", non_empty)


CAPS
Empty: 12981
Non-empty: 7019

TTPS
Empty: 13724
Non-empty: 6276

MBC
Empty: 13409
Non-empty: 6591


In [58]:
for col in ['caps', 'ttps', 'mbc']:

    print(f"\n{col.upper()}")

    result = dataset_1.groupby('sha256')[col].apply(
        lambda x: [
            len(v) for v in x
        ]
    )

    print(result.head(10))


CAPS
sha256
00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2    [0, 30]
0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9eaf94f82d2eb1dd0291    [25, 0]
001aea7ddef769f8b44ffc497cdf4f05193b4889510a5385a5d778817a09f8c3     [4, 0]
001b30621be4731fbd66ce0480ffcf15f8feeebedd9b159352c24ff146b718cc     [1, 0]
001dc027ca8aea62fdf19327d2a74ddf156aef93c8cbb0dab6b4ba728d475bc9     [2, 0]
001e3da23ae36c7ef6af5850f469234b06fc40779ec8f43749465ec0351ed467     [5, 0]
00208499e4f2a02320ebdd993446ecd310b026e19cfebb38eec60b73dcb5815f    [0, 29]
00215869134d30dbe58d0c9c3b462ac33590b0552bf71af964a9a76842e0163a    [0, 27]
0022831516cb4d9e122c17cba922efdccfef0ceaaea9d4eef34a629574d35cc3    [31, 0]
0022df1d0967d5de37784ae774e7447b6327fa509aeb87a9084aa4f371921c0f     [2, 0]
Name: caps, dtype: object

TTPS
sha256
00047dd8d224e115c979a9eb1acd116e7a4a8a72f025941876b8776bea2c39e2    [0, 14]
0006d1a82c9c2ad4dfbbe34772b104bc889b4c0ebdcc9eaf94f82d2eb1dd0291    [11, 0]
001aea7ddef769f8b44ffc497cdf4f05193b

In [59]:
for col in ['caps', 'ttps', 'mbc']:

    stats = dataset_1.groupby('sha256')[col].apply(
        lambda x: sum(len(v) > 0 for v in x)
    )

    print(f"\n{col.upper()}")
    print("Both empty:", (stats == 0).sum())
    print("One enriched:", (stats == 1).sum())
    print("Both enriched:", (stats == 2).sum())


CAPS
Both empty: 2981
One enriched: 7019
Both enriched: 0

TTPS
Both empty: 3724
One enriched: 6276
Both enriched: 0

MBC
Both empty: 3409
One enriched: 6591
Both enriched: 0


In [60]:
# Check all three enrichment fields together

summary = dataset_1.groupby('sha256').agg(
    caps_count=('caps', lambda x: sum(len(v) > 0 for v in x)),
    ttps_count=('ttps', lambda x: sum(len(v) > 0 for v in x)),
    mbc_count=('mbc', lambda x: sum(len(v) > 0 for v in x))
)

all_three = (
    (summary['caps_count'] == 1) &
    (summary['ttps_count'] == 1) &
    (summary['mbc_count'] == 1)
)

print("All three enriched:", all_three.sum())

print("At least one enriched:",
      ((summary[['caps_count', 'ttps_count', 'mbc_count']] > 0)
       .any(axis=1)).sum())

print("All three empty:",
      ((summary[['caps_count', 'ttps_count', 'mbc_count']] == 0)
       .all(axis=1)).sum())

All three enriched: 6085
At least one enriched: 7042
All three empty: 2958


In [63]:
import pandas as pd
import hashlib
import json

def normalize_value(x):

    if isinstance(x, (list, dict)):
        return json.dumps(
            x,
            sort_keys=True,
            default=str
        )

    if pd.isna(x):
        return "<NULL>"

    return str(x)


conflict_results = {}

for col in dataset_1.columns:

    if col == 'sha256':
        continue

    temp = dataset_1[['sha256', col]].copy()

    temp['_normalized'] = temp[col].apply(
        normalize_value
    )

    conflicts = (
        temp.groupby('sha256')['_normalized']
        .nunique()
    )

    count = (conflicts > 1).sum()

    conflict_results[col] = count

    if count > 0:
        print(f"{col}: {count} conflicting groups")

print("\nTotal columns checked:", len(conflict_results))

caps: 7019 conflicting groups
ttps: 6276 conflicting groups
mbc: 6591 conflicting groups

Total columns checked: 31


In [64]:
print("Shape:", dataset_1.shape)

print("\nColumns:")
for i, col in enumerate(dataset_1.columns, 1):
    print(i, col)

print("\nFile types:")
print(dataset_1['file_type'].value_counts(dropna=False))

print("\nLabels:")
print(dataset_1['label'].value_counts(dropna=False))

Shape: (20000, 32)

Columns:
1 md5
2 sha1
3 sha256
4 tlsh
5 first_submission_date
6 last_analysis_date
7 detection_ratio
8 label
9 file_type
10 family
11 family_confidence
12 behavior
13 file_property
14 packer
15 exploit
16 group
17 histogram
18 byteentropy
19 strings
20 general
21 header
22 section
23 imports
24 exports
25 datadirectories
26 richheader
27 authenticode
28 pefilewarnings
29 week_id
30 caps
31 ttps
32 mbc

File types:
file_type
Win64    20000
Name: count, dtype: int64

Labels:
label
1    10000
0    10000
Name: count, dtype: int64
